# Bulk Expression Viewer for Colab

This notebook is the Colab-native entry point for colleagues.

Recommended order:
1. Run the clone form.
2. Run the Google Drive data download form.
3. Run the setup/import form.
4. Run the selection form.
5. Run the plotting form.
6. Run the export form if you want CSV and PNG outputs.


In [ ]:
# @title Clone repository
repo_url = "https://github.com/dobereiner/bulk_expression.git"  # @param {type:"string"}
repo_branch = "master"  # @param {type:"string"}
repo_path = "/content/bulk_expression"  # @param {type:"string"}
force_reclone = False  # @param {type:"boolean"}

import shutil
import subprocess
from pathlib import Path

repo_root = Path(repo_path).expanduser().resolve()
if force_reclone and repo_root.exists():
    shutil.rmtree(repo_root)

if repo_root.exists():
    print(f"Repository already exists: {repo_root}")
else:
    subprocess.check_call([
        "git", "clone", "--branch", repo_branch, repo_url, str(repo_root)
    ])
    print(f"Repository cloned to: {repo_root}")


In [ ]:
# @title Download data folder from Google Drive
repo_path = "/content/bulk_expression"  # @param {type:"string"}
drive_folder_id = "1dr5F5TPhdMjU9ueUvGoQUDi0qSvO2YkV"  # @param {type:"string"}
replace_existing_data = False  # @param {type:"boolean"}

import shutil
import subprocess
import sys
from pathlib import Path

repo_root = Path(repo_path).expanduser().resolve()
if not repo_root.exists():
    raise FileNotFoundError(f"Repository folder not found: {repo_root}. Run the clone cell first.")

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown>=5.2,<6.0"])
    import gdown

data_dir = repo_root / "data"
download_root = repo_root / "_gdrive_download"

if replace_existing_data and data_dir.exists():
    shutil.rmtree(data_dir)

if download_root.exists():
    shutil.rmtree(download_root)
download_root.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

downloaded = gdown.download_folder(
    id=drive_folder_id,
    output=str(download_root),
    quiet=True,
    use_cookies=False,
    remaining_ok=True,
)

children = list(download_root.iterdir())
source_root = children[0] if len(children) == 1 and children[0].is_dir() else download_root

for item in source_root.iterdir():
    destination = data_dir / item.name
    if item.is_dir():
        shutil.copytree(item, destination, dirs_exist_ok=True)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, destination)

from IPython.display import Markdown, display
display(Markdown(
    f"**Data copied to:** `{data_dir}`  \n"
    f"**Downloaded files:** {0 if downloaded is None else len(downloaded)}  \n"
    f"**Top-level contents:** {', '.join(sorted(p.name for p in data_dir.iterdir()))}"
))


In [ ]:
# @title Setup repository and import libraries
repo_path = "/content/bulk_expression"  # @param {type:"string"}
install_requirements = True  # @param {type:"boolean"}

import os
import subprocess
import sys
from pathlib import Path

repo_root = Path(repo_path).expanduser().resolve()
if not repo_root.exists():
    raise FileNotFoundError(
        f"Repository folder not found: {repo_root}. Clone the repo first, then rerun this cell."
    )

os.chdir(repo_root)
if install_requirements:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

from IPython.display import Markdown, display

sys.path.insert(0, str((repo_root / "src").resolve()))

from bulk_expression_viewer import (
    export_render_bundle,
    get_scale_matrix,
    load_project_data,
    matrix_to_long,
    plot_aggregated_heatmap,
    plot_gene_barplot,
    plot_multi_gene_heatmap,
    plot_passage_trend,
    resolve_genes,
    parse_gene_input,
    parse_sample_input,
    setup_plot_style,
)

setup_plot_style()
project = load_project_data(repo_root / "data" / "processed")

display(Markdown(
    f"**Repository ready:** `{repo_root}`  \n"
    f"**Loaded genes:** {project.tpm.shape[0]:,}  \n"
    f"**Loaded samples:** {project.tpm.shape[1]}"
))
display(project.metadata[["sample_id", "sample_name", "line", "condition", "project_group", "passage"]].head(10))


In [ ]:
# @title Choose genes and samples
genes = "EPCAM, VIM, MKI67"  # @param {type:"string"}
scale = "TPM"  # @param ["counts", "TPM", "log2(TPM+1)", "VST"]
sample_mode = "All samples"  # @param ["All samples", "Project group", "Condition", "Custom sample numbers"]
project_group = "All"  # @param ["All", "PRC", "CC"]
condition_filter = "All"  # @param ["All", "baseline", "full", "-pp", "-nog", "-rspo", "-nog -rspo"]
line_filter = ""  # @param {type:"string"}
custom_sample_numbers = ""  # @param {type:"string"}
heatmap_clip_quantile = 0.99  # @param [0.95, 0.98, 0.99]
aggregate_enabled = True  # @param {type:"boolean"}
aggregate_by = "line"  # @param ["line", "condition", "project_group", "passage"]
aggregate_function = "mean"  # @param ["mean", "median"]
preview_rows = 25  # @param {type:"slider", min:5, max:100, step:5}

requested_genes = parse_gene_input(genes)
if not requested_genes:
    raise ValueError("Please provide at least one gene symbol.")

selected_meta = project.metadata.copy()
if sample_mode == "Custom sample numbers":
    sample_keys = parse_sample_input(custom_sample_numbers, project.metadata["sample_id"].tolist())
    selected_meta = selected_meta[selected_meta["sample_key"].isin(sample_keys)]

if project_group != "All":
    selected_meta = selected_meta[selected_meta["project_group"] == project_group]

if condition_filter != "All":
    selected_meta = selected_meta[selected_meta["condition"] == condition_filter]

if line_filter.strip():
    requested_lines = [item.strip() for item in line_filter.split(',') if item.strip()]
    selected_meta = selected_meta[selected_meta["line"].isin(requested_lines)]

selected_meta = selected_meta.sort_values("sample_id")
if selected_meta.empty:
    raise ValueError("The current filters selected zero samples.")

matrix = get_scale_matrix(scale, project)
resolved_genes, suggestions = resolve_genes(requested_genes, matrix.index)
if not resolved_genes:
    suggestion_lines = []
    for gene, matches in suggestions.items():
        if matches:
            suggestion_lines.append(f"- `{gene}`: try {', '.join(matches)}")
    suggestion_text = "\n".join(suggestion_lines) if suggestion_lines else "No close matches found."
    raise ValueError(f"No gene symbols were resolved.\n\n{suggestion_text}")

sample_keys = selected_meta["sample_key"].tolist()
matrix_subset = matrix.loc[resolved_genes, sample_keys]
long_df = matrix_to_long(matrix, selected_meta, resolved_genes, sample_keys)
preview_df = long_df[["gene_symbol", "sample_id", "sample_name", "line", "condition", "project_group", "passage", "expression"]].sort_values(["gene_symbol", "sample_id"])

summary_md = (
    f"**Resolved genes:** {', '.join(resolved_genes)}  \n"
    f"**Samples shown:** {len(sample_keys)}  \n"
    f"**Scale:** `{scale}`  \n"
    f"**Filters:** project_group=`{project_group}`, condition=`{condition_filter}`, sample_mode=`{sample_mode}`  \n"
    f"**Heatmap clipping quantile:** q={heatmap_clip_quantile}  \n"
    f"**Project groups present:** {', '.join(sorted(selected_meta['project_group'].astype(str).unique()))}"
)

display(Markdown(summary_md))
unresolved = [gene for gene in requested_genes if gene not in resolved_genes]
if unresolved:
    unresolved_lines = []
    for gene in unresolved:
        matches = suggestions.get(gene, [])
        unresolved_lines.append(f"- `{gene}`: {', '.join(matches) if matches else 'no close match'}")
    display(Markdown("**Unresolved genes**\n" + "\n".join(unresolved_lines)))

display(preview_df.head(preview_rows).reset_index(drop=True))


In [ ]:
# @title Plot selected expression
show_preview_tab = True  # @param {type:"boolean"}
z_scale_by_gene = True  # @param {type:"boolean"}

import matplotlib.pyplot as plt

effective_heatmap_transform = "Row z-score by gene" if z_scale_by_gene else "Quantile clip"

transform_lookup = {
    "Quantile clip": "quantile_clip",
    "Row z-score by gene": "row_zscore",
}
heatmap_transform_key = transform_lookup[effective_heatmap_transform]

try:
    from google.colab import widgets as colab_widgets
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

figures = {}

tab_names = []
if show_preview_tab:
    tab_names.append("Summary")
if len(resolved_genes) == 1:
    tab_names.append("Barplot")
    if long_df["passage"].notna().sum() >= 2:
        tab_names.append("Passage")
tab_names.append("Heatmap")
if aggregate_enabled:
    tab_names.append("Aggregated")

if IN_COLAB:
    tabbar = colab_widgets.TabBar(tab_names)
    tab_index = 0

    if show_preview_tab:
        with tabbar.output_to(tab_index):
            display(Markdown(summary_md))
            display(preview_df.head(preview_rows).reset_index(drop=True))
        tab_index += 1

    if len(resolved_genes) == 1:
        with tabbar.output_to(tab_index):
            fig = plot_gene_barplot(long_df, scale)
            figures["sample_barplot"] = fig
            display(fig)
            plt.close(fig)
        tab_index += 1

        if long_df["passage"].notna().sum() >= 2:
            with tabbar.output_to(tab_index):
                fig = plot_passage_trend(long_df, scale)
                figures["passage_trend"] = fig
                display(fig)
                plt.close(fig)
            tab_index += 1

    with tabbar.output_to(tab_index):
        fig = plot_multi_gene_heatmap(matrix_subset, selected_meta, scale, transform=heatmap_transform_key, clip_quantile=float(heatmap_clip_quantile))
        figures["heatmap"] = fig
        display(fig)
        plt.close(fig)
    tab_index += 1

    if aggregate_enabled:
        with tabbar.output_to(tab_index):
            fig = plot_aggregated_heatmap(long_df, aggregate_by, aggregate_function, scale, transform=heatmap_transform_key, clip_quantile=float(heatmap_clip_quantile))
            figures["aggregated_heatmap"] = fig
            display(fig)
            plt.close(fig)
else:
    display(Markdown(summary_md))
    if len(resolved_genes) == 1:
        fig = plot_gene_barplot(long_df, scale)
        figures["sample_barplot"] = fig
        display(fig)
        plt.close(fig)

        if long_df["passage"].notna().sum() >= 2:
            fig = plot_passage_trend(long_df, scale)
            figures["passage_trend"] = fig
            display(fig)
            plt.close(fig)

    fig = plot_multi_gene_heatmap(matrix_subset, selected_meta, scale, transform=heatmap_transform_key, clip_quantile=float(heatmap_clip_quantile))
    figures["heatmap"] = fig
    display(fig)
    plt.close(fig)

    if aggregate_enabled:
        fig = plot_aggregated_heatmap(long_df, aggregate_by, aggregate_function, scale, transform=heatmap_transform_key, clip_quantile=float(heatmap_clip_quantile))
        figures["aggregated_heatmap"] = fig
        display(fig)
        plt.close(fig)


In [ ]:
# @title Export current selection
export_bundle = False  # @param {type:"boolean"}
export_dir = "data/exports"  # @param {type:"string"}

if export_bundle:
    bundle_dir = export_render_bundle(
        long_df=preview_df,
        matrix_subset=matrix_subset,
        metadata_subset=selected_meta,
        figures=figures,
        output_dir=repo_root / export_dir,
    )
    display(Markdown(f"**Exported to:** `{bundle_dir}`"))
else:
    display(Markdown("Set `export_bundle=True` and rerun this cell to save the current selection."))
